# 📚 Notebook 1: What is RAG — and Why Do We Need It?

**Learning Goal:** Understand *why* RAG exists, and see the difference it makes with your own eyes.

---

## The Problem We're Solving

You spent years studying for your MBA. You have 1,250 files — case studies, financial models, research papers, assignments. That's a huge knowledge base.

Now you want to ask questions like:
- *"What case studies did I study about disruption?"*
- *"Summarize what I learned about Porter's Five Forces"*
- *"What was the financial model I built for the Reliance Industries case?"*

**The catch:** GPT-4 doesn't know your files. It was trained on public internet data, not your personal MBA documents.

If you ask GPT-4 about your MBA files, it will either:
1. Give a generic answer (not specific to your program)
2. **Hallucinate** — make up plausible-sounding but wrong information

---

## The Solution: RAG

**RAG = Retrieval Augmented Generation**

In plain English: *Find the relevant parts of your files, then ask GPT-4 to answer using those parts.*

```
Without RAG:  Your Question ──────────────────────────→ GPT-4 → Generic/Wrong Answer

With RAG:     Your Question → [Find relevant chunks] → GPT-4 → Accurate Answer + Sources
                                from your 1,250 files     ↑
                                                    your files become
                                                    GPT-4's "memory"
```

---

## What You'll Do in This Notebook

1. ✅ Set up your OpenAI API key
2. 🚫 Ask GPT-4 about your MBA **without** RAG → see the problem
3. ✅ Ask the same question **with** RAG context → see the solution
4. 🗺️ Understand the 6-step pipeline we'll build in Notebooks 2-6

**Time needed:** ~15 minutes | **Cost:** < $0.001

---
## Step 1: Install the OpenAI Library

**What this does:** Downloads the Python package that lets us talk to OpenAI's GPT-4.

▶ **Press the play button** on the cell below. Wait for it to finish (you'll see a ✅ when done).

In [ ]:
#@title ⚙️ Step 1 — Install OpenAI library  { display-mode: "form" }
!pip install openai -q
print("✅ OpenAI library installed successfully!")

---
## Step 2: Load Your OpenAI API Key

**What is an API key?** It's like a password that tells OpenAI which account to charge. It looks like: `sk-proj-abc123...`

**Where to get it:** [platform.openai.com/api-keys](https://platform.openai.com/api-keys)

### Running on the website?
Click **"Run in browser"** in the toolbar above and enter your key there before running cells.
The website will set it automatically — you don't need to do anything in this cell.

### Running in Google Colab?
Add your key via **Colab Secrets** (the 🔑 icon in the left sidebar):
1. Name: `OPENAI_API_KEY`
2. Value: your actual key (`sk-proj-abc123...`)
3. Toggle **Notebook access** to ON

The cell below loads the key automatically from whichever environment you're in.

In [ ]:
#@title 🔑 Step 2 — Load your API key  { display-mode: "form" }
import os
import openai

OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', '')

if not OPENAI_API_KEY:
    try:
        from google.colab import userdata
        OPENAI_API_KEY = userdata.get('OPENAI_API_KEY') or ''
        if OPENAI_API_KEY:
            os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
    except Exception:
        pass

if OPENAI_API_KEY:
    print(f"✅ API key loaded. Starts with: {OPENAI_API_KEY[:8]}...")
else:
    print("⚠️  API key not found.")
    print("   Open the 🔑 Secrets panel (left sidebar) → Add secret:")
    print("   Name: OPENAI_API_KEY  |  Value: your key (sk-proj-...)")
    print("   Then re-run this cell.")

client = openai.OpenAI(api_key=OPENAI_API_KEY)
print()
print("✅ Ready to use GPT-4!")

---
## Part 1: Ask GPT-4 WITHOUT Your MBA Files

Let's first see what GPT-4 says when it has **no access to your files**.

We'll ask: *"What case studies did I study in my MBA program about competitive strategy?"*

GPT-4 has never seen your files, so watch what happens...

In [ ]:
#@title 🔧 Background setup — run once, no output expected  { display-mode: "form" }
def ask_without_rag(question):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": question}
        ]
    )
    return response.choices[0].message.content

print("✅ Functions ready.")

In [ ]:
#@title 🚫 Demo 1 — Ask GPT-4 WITHOUT your files  { display-mode: "form" }
#@markdown Type a question about your MBA below and run this cell. Watch GPT-4 give a **generic answer** — it has never seen your files.
question = "What case studies did I study in my MBA program about competitive strategy?" #@param {type:"string"}

print(f"❓ Question: {question}")
print()
print("⏳ Asking GPT-4 (no context from your files)...")
print()

answer = ask_without_rag(question)

print("💬 GPT-4's answer (without RAG):")
print("─" * 60)
print(answer)
print("─" * 60)
print()
print("⚠️  Notice: This is a GENERIC answer.")
print("   GPT-4 doesn't know what YOU specifically studied.")
print("   It may even mention case studies you never read!")

---
## Part 2: Ask GPT-4 WITH Context from Your Files

Now let's give GPT-4 some context — specific text extracted from your actual MBA files.

**Important:** In this notebook, we're providing the context manually just to show the concept.
In Notebooks 2-6, we'll build the system that retrieves this context **automatically** for any question.

Watch how GPT-4's answer changes when it can *see* your actual documents...

In [ ]:
#@title 🔧 Background setup — RAG function (run once)  { display-mode: "form" }
def ask_with_rag(question, context):
    rag_prompt = f"""You are an MBA knowledge assistant helping the user recall what they studied.

Use ONLY the information in the CONTEXT section below to answer the question.
If the answer is not found in the context, say: "I don't have that information in the provided documents."
Do NOT make up or add any information that isn't in the context.
Always mention which source file(s) the answer came from.

CONTEXT (extracted from your MBA files):
─────────────────────────────────────────
{context}
─────────────────────────────────────────

QUESTION: {question}

ANSWER:"""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": rag_prompt}]
    )
    return response.choices[0].message.content

print("✅ RAG function ready.")

In [ ]:
#@title ✅ Demo 2 — Ask the SAME question WITH context from your files  { display-mode: "form" }
#@markdown Watch how GPT-4's answer completely changes when it can **read your actual documents** and cite sources.

# Sample context — in the real system this is retrieved automatically from your 1,250 files
sample_context = """
Source: Strategy/Porter_Five_Forces_Apple_Case.pdf  (Page 3)
─────────────────────────────────────────────────────────────
This case study examined Apple Inc. through Porter's Five Forces framework.
The analysis showed strong barriers to entry due to Apple's brand equity and
ecosystem lock-in. Threat of substitutes was low as iOS alternatives required
complete ecosystem migration. Competitive rivalry was intense with Samsung
and Google. The case concluded that Apple's sustained competitive advantage
comes from vertical integration and ecosystem switching costs.

Source: Strategy/Blue_Ocean_Nintendo_Wii.pdf  (Page 7)
─────────────────────────────────────────────────────────────
Nintendo's Wii launch in 2006 was studied as a textbook example of Blue Ocean
Strategy. Instead of competing with Sony PS3 and Microsoft Xbox 360 on
processing power, Nintendo created a new market space by targeting non-gamers
with intuitive motion-based controls — value innovation at lower cost.

Source: Strategy/Competitive_Dynamics_Airlines.pdf  (Page 2)
─────────────────────────────────────────────────────────────
The airline industry case examined competitive dynamics using the
Awareness-Motivation-Capability (AMC) framework. Southwest Airlines was
analysed as a case of strategic commitment — its low-cost model created
asymmetric competitive responses where legacy carriers could not match fares.
"""

print(f"❓ Same question: {question}")
print()
print("⏳ Asking GPT-4 WITH context from your MBA files...")
print()

answer_with_rag = ask_with_rag(question, sample_context)

print("💬 GPT-4's answer (with RAG):")
print("─" * 60)
print(answer_with_rag)
print("─" * 60)
print()
print("✅ Notice the difference:")
print("   1. Answer is based on YOUR specific files")
print("   2. It cites the actual source PDFs")
print("   3. It won't invent case studies you never read")

---
## Part 3: Side-by-Side Comparison

Let's put both answers side by side to clearly see the RAG difference.

In [ ]:
#@title 📊 Side-by-side comparison  { display-mode: "form" }
print("=" * 70)
print("COMPARISON: Same Question, Two Approaches")
print("=" * 70)
print()
print("🚫 WITHOUT RAG (GPT-4 guessing from training data):")
print("─" * 70)
print(answer[:400] + "...")
print()
print("✅ WITH RAG (GPT-4 reading your actual MBA files):")
print("─" * 70)
print(answer_with_rag)
print()
print("=" * 70)
print("Without RAG:  Generic, could be wrong, no source")
print("With RAG:     Specific to your files, cites sources, grounded")
print("=" * 70)

---
## Part 4: The 6-Step Pipeline We're Building

In the demo above, we *manually* provided the context. But with 1,250 files,
you can't manually find relevant text every time you ask a question.

The remaining 5 notebooks build a system that does this **automatically** — for any question, across all your files, in seconds.

Here's the complete pipeline:

In [ ]:
#@title 🗺️ The complete 6-step pipeline  { display-mode: "form" }
pipeline = """
╔══════════════════════════════════════════════════════════════╗
║           THE 6-STEP RAG PIPELINE YOU'RE BUILDING           ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  ONE-TIME SETUP (run once, takes ~20 mins for 1,250 files)  ║
║  ─────────────────────────────────────────────────────────  ║
║  [1] PARSE   → Read all PDF/DOCX/PPTX/XLSX files            ║
║                            ↓                                ║
║  [2] CHUNK   → Split long texts into 500-word pieces        ║
║                            ↓                                ║
║  [3] EMBED   → Convert each chunk to 1,536 numbers          ║
║               Cost: ~$0.20–$0.30 total, one-time            ║
║                            ↓                                ║
║  [4] STORE   → Save all chunks + numbers to a file          ║
║                                                              ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  EVERY QUESTION (runs in ~2 seconds, costs ~$0.001)         ║
║  ─────────────────────────────────────────────────────────  ║
║  [5] SEARCH  → Find 5 chunks most similar to your question  ║
║                            ↓                                ║
║  [6] ANSWER  → GPT-4 answers using ONLY your documents      ║
║               Returns answer + source file names            ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
"""
print(pipeline)

In [ ]:
#@title 📚 How each notebook maps to the pipeline  { display-mode: "form" }
roadmap = """
┌────────────────┬────────────────────────────────────────────────────┐
│ Notebook       │ What You'll Learn                                  │
├────────────────┼────────────────────────────────────────────────────┤
│ 01 (THIS ONE)  │ Why RAG exists, the problem it solves              │
│ 02_parse       │ Step 1: Read PDF/DOCX/PPTX/XLSX → extract text    │
│ 03_chunk       │ Step 2: Split text into 500-word chunks            │
│ 04_embeddings  │ Step 3: Convert text to numbers (vectors)          │
│ 05_retrieval   │ Step 5: Find relevant chunks using math            │
│ 06_pipeline    │ Step 6: Full system — ask any question             │
└────────────────┴────────────────────────────────────────────────────┘
"""
print(roadmap)

---
## Part 5: Key Concepts Glossary

These terms will appear throughout all 6 notebooks. Bookmark this section.

In [ ]:
#@title 📖 Key terms glossary  { display-mode: "form" }
glossary = [
    ("Embedding",         "A list of ~1,536 numbers that capture the MEANING of text. Similar meanings → similar numbers."),
    ("Vector",            "Another word for 'embedding' — a list of numbers."),
    ("Chunk",             "A 500-word piece of text cut from your documents."),
    ("Cosine Similarity", "Math formula: how 'close' are two meanings? Score 0 = unrelated. Score 1 = identical."),
    ("Vector Store",      "The file/database where all your chunks + embeddings live."),
    ("Retrieval",         "Finding the chunks most relevant to your question."),
    ("Context",           "The retrieved chunks sent to GPT-4 so it can answer accurately."),
    ("RAG",               "Retrieve relevant chunks → Augment the prompt → Generate answer."),
    ("Hallucination",     "When an AI makes up plausible-sounding but wrong information."),
]

print("RAG GLOSSARY")
print("=" * 70)
for term, definition in glossary:
    print(f"\n  {term:<20} {definition}")
print("\n" + "=" * 70)

---
## Bonus: Test the Anti-Hallucination Guard

One of the most important features of our RAG system is that it **refuses to answer** when the answer isn't in your files — instead of making something up.

Let's test this by asking about something that would NOT be in the sample context we provided.

In [ ]:
#@title 🛡️ Bonus — Test the anti-hallucination guard  { display-mode: "form" }
#@markdown Ask about something **not in the sample context** — the system should say "I don't have that" instead of making something up.
out_of_scope_question = "What financial models did I build for Reliance Industries?" #@param {type:"string"}

print(f"❓ Question: {out_of_scope_question}")
print()
print("⏳ Asking with RAG (using the same sample context from Demo 2)...")
print()

out_of_scope_answer = ask_with_rag(out_of_scope_question, sample_context)

print("💬 GPT-4's answer:")
print("─" * 60)
print(out_of_scope_answer)
print("─" * 60)
print()
print("✅ The system correctly says it doesn't have the information.")
print("   It did NOT make up a fake answer — that's the anti-hallucination guard working.")

---
## ✅ Notebook 1 Complete!

### What you learned:

1. **The problem**: GPT-4 doesn't know your files → generic or hallucinated answers
2. **The solution**: RAG — retrieve relevant text, then ask GPT-4 to reason over it
3. **The difference**: With RAG, answers are specific, sourced, and grounded
4. **The safeguard**: Anti-hallucination prompt makes GPT-4 say "I don't know" instead of guessing
5. **The pipeline**: 6 steps — 4 one-time setup steps + 2 per-question steps

### The key insight:
> RAG doesn't make GPT-4 smarter. It gives GPT-4 the **right documents to read** before answering.
> You're turning GPT-4 from a guesser into a reader of YOUR documents.

---

### Next: Notebook 2 — Document Parsing

In Notebook 2, you'll learn how Python reads your actual MBA files:
- How it extracts text from PDFs page by page
- How it reads DOCX paragraphs, PPTX slide text, XLSX cell values
- How your 30 subject folders become automatic tags
- What happens to images (they're skipped for now)

**Before starting Notebook 2:**
1. Update your progress in README.md — check off Notebook 1
2. Upload 5-10 MBA files to your `data/sample/` folder in Google Drive
   - Try to include: 1 PDF, 1 DOCX, 1 PPTX, 1 XLSX
   - Put them in subfolders matching your subject (e.g., `data/sample/Finance/`)